## 1.0 Set environment path
+ To configure the Python and Java environment required for PySpark to run correctly before creating a Spark session

In [13]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"

## 2.0 Create spark session
+ SparkSession is the entry point to use Spark functionality in Python

In [14]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IrisFix") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print("Spark Ready!")

Spark Ready!


In [15]:
## Spark test command
# Function: verify that Apache Spark is installed correctly and running properly
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



## 3.0 Load dataset
+ The link:
  + https://raw.githubusercontent.com/uiuc-cse/data-fa14/gh-pages/data/iris.csv
+ Iris dataset consists of 150 observations of iris flowers with four numerical features:
  + Sepal length
  + Sepal width
  + Petal length
  + Petal width
+ Target variable is the species, which has 3 classes:
  + Setosa
  + Versicolor
  + Virginica

In [16]:
iris = spark.read.csv(
    "iris.csv",
    header=True,
    inferSchema=True
)

iris.show(5)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows



## 4.0 Data preprocessing

#### 4.1 Convert categorical species into numeric labels
+ Using StringIndexer to transform categorical class values in the species column into numerical labels stored in the label column, enabling compatibility with supervised machine learning algorithms

In [17]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="species", outputCol="label")
iris = indexer.fit(iris).transform(iris)

#### 4.2 Assemble feature columns
+ Use VectorAssembler to merge multiple predictor variables into a single feature vector column named features, which is the standard input structure required by PySpark MLlib classification algorithms

In [18]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="features"
)

iris = assembler.transform(iris)
iris.select("features", "label").show(5)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|  0.0|
|[4.9,3.0,1.4,0.2]|  0.0|
|[4.7,3.2,1.3,0.2]|  0.0|
|[4.6,3.1,1.5,0.2]|  0.0|
|[5.0,3.6,1.4,0.2]|  0.0|
+-----------------+-----+
only showing top 5 rows



#### 4.3 Check class distribution
+ Purpose: To assess the class distribution, enabling the identification of potential class imbalance that may affect the performance and reliability of machine learning

In [19]:
iris.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0|   50|
|  1.0|   50|
|  2.0|   50|
+-----+-----+



## 5.0 Train-Test split
+ Split the dataset into training and testing data for machine learning model development and evaluation
+ Divide the dataset into:
  + Training data (80%): Used to train/ learn the model
  + Testing data (20%): Used to evaluate model performance on unseen data
  + This helps measure how well the model generalizes

In [20]:
train_data, test_data = iris.randomSplit([0.8, 0.2], seed=123)

## 6.0 Classification model

#### Model 1 - Logistic regression
+ Build a Logistic Regression classifier and applies hyperparameter tuning using a 5-fold cross validation approach.
+ It evaluates multiple parameter combinations using accuracy as the metric and selects the best-performing model, which is then used to make predictions on unseen data

In [21]:
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

log = LogisticRegression(featuresCol="features", labelCol="label")

paramGrid_lr = (ParamGridBuilder()
                .addGrid(log.regParam, [0.01, 0.1, 1])
                .addGrid(log.maxIter, [10, 20])
                .build())

metric = MulticlassClassificationEvaluator(metricName="accuracy")

cv_log = CrossValidator(estimator=log,
                       estimatorParamMaps=paramGrid_lr,
                       evaluator=metric,
                       numFolds=5)

log_model = cv_log.fit(train_data)
log_predictions = log_model.transform(test_data)

#### Model 2 - Decision tree
+ Build a Decision Tree classifier and applies hyperparameter tuning using cross-validation.
+ It tests multiple values of max depth and minimum instances per node, evaluates performance using accuracy and select the best-performing model to make predictions on unseen test data

In [22]:
tree = DecisionTreeClassifier(featuresCol="features", labelCol="label")

paramGrid_tree = (ParamGridBuilder()
                .addGrid(tree.maxDepth, [3,5,7])
                .addGrid(tree.minInstancesPerNode, [1,2,4])
                .build())

cv_tree = CrossValidator(estimator=tree,
                       estimatorParamMaps=paramGrid_tree,
                       evaluator=metric,
                       numFolds=5)

tree_model = cv_tree.fit(train_data)
tree_predictions = tree_model.transform(test_data)

#### Model 3 - Random forest
+ Built a Random Forest classifier and performs hyperparameter tuning using cross validation.
+ It evaluates different combinations of number of trees and maximum tree depth, select the best-performing model based on accuracy, and applies it to predict unseen test data

In [23]:
forest = RandomForestClassifier(featuresCol="features", labelCol="label")

paramGrid_forest = (ParamGridBuilder()
                .addGrid(forest.numTrees, [10,20,50])
                .addGrid(forest.maxDepth, [5,10])
                .build())

cv_forest = CrossValidator(estimator=forest,
                       estimatorParamMaps=paramGrid_forest,
                       evaluator=metric,
                       numFolds=5)

forest_model = cv_forest.fit(train_data)
forest_predictions = forest_model.transform(test_data)

## 7.0 Model evaluation

#### Explanation of each metric:
+ Accuracy:
  + Measures the proportion of correctly classified Iris samples across all three species.
  + Accuracy = $\frac{TP + TN}{TP + TN + FP + FN}$
+ Weighted precision:
  + Evaluates how accurate the model's predictions for each species, then averages them while considering the number of samples per class.
  + Precision = $\frac{TP}{TP + FP}$
+ Weighted recall:
  + Measure how well the model identifies all actual instances of each Iris species.
  + Recall = $\frac{TP}{TP + FN}$
+ F1-Score:
  + Provides a balance between precision and recall for all three species.
  + F1 = $2 \times \frac{Precision \times Recall}{Precision + Recall}$
 
These metrics are used to determine how effectively the model distinguishes between the three Iris species. Since the Iris dataset is typically balanced, the evaluation metrics should be relatively consistent. High values across all metrics indicate that the model can accurately classify flower species based on their features.

In [24]:
accuracy = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy")

precision = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision")

recall = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall")

f1_score = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1")

print("Logistic Regression Metrics:")
print("Accuracy:", accuracy.evaluate(log_predictions))
print("Precision:", precision.evaluate(log_predictions))
print("Recall:", recall.evaluate(log_predictions))
print("F1-score:", f1_score.evaluate(log_predictions))

Logistic Regression Metrics:
Accuracy: 0.9655172413793104
Precision: 0.9689655172413794
Recall: 0.9655172413793104
F1-score: 0.9650984224486947


In [28]:
print("Decision Tree Metrics:")
print("Accuracy:", accuracy.evaluate(tree_predictions))
print("Precision:", precision.evaluate(tree_predictions))
print("Recall:", recall.evaluate(tree_predictions))
print("F1-score:", f1_score.evaluate(tree_predictions))

Decision Tree Metrics:
Accuracy: 0.9310344827586207
Precision: 0.9310344827586207
Recall: 0.9310344827586207
F1-score: 0.9310344827586207


In [29]:
print("Random Forest Metrics:")
print("Accuracy:", accuracy.evaluate(forest_predictions))
print("Precision:", precision.evaluate(forest_predictions))
print("Recall:", recall.evaluate(forest_predictions))
print("F1-score:", f1_score.evaluate(forest_predictions))

Random Forest Metrics:
Accuracy: 0.9655172413793104
Precision: 0.9689655172413794
Recall: 0.9655172413793104
F1-score: 0.9650984224486947


## 8.0 Generate predictions

+ This step generats predicted class labels for each observation in the testing dataset using trained machine learning models. The output displays the feature vectors (features), actual class labels (label), and model predicted labels (prediction) for the Iris dataset.
+ The results allow direct comparison between actual and predicted classes, enabling evaluation of model performance in classifying Iris species. This comparison is used to assess and benchmark different classifiers such as Logistic Regression, Random Forest and Decision Tree.

#### 8.1 Logistic regression prediction

In [31]:
log_predictions.select("features","label","prediction").show(10)

+-----------------+-----+----------+
|         features|label|prediction|
+-----------------+-----+----------+
|[4.4,3.0,1.3,0.2]|  0.0|       0.0|
|[4.6,3.2,1.4,0.2]|  0.0|       0.0|
|[4.8,3.0,1.4,0.3]|  0.0|       0.0|
|[4.8,3.1,1.6,0.2]|  0.0|       0.0|
|[4.9,3.0,1.4,0.2]|  0.0|       0.0|
|[5.0,2.3,3.3,1.0]|  1.0|       1.0|
|[5.0,3.5,1.3,0.3]|  0.0|       0.0|
|[5.0,3.5,1.6,0.6]|  0.0|       0.0|
|[5.1,3.3,1.7,0.5]|  0.0|       0.0|
|[5.1,3.4,1.5,0.2]|  0.0|       0.0|
+-----------------+-----+----------+
only showing top 10 rows



#### 8.2 Decision tree prediction

In [32]:
tree_predictions.select("features","label","prediction").show(10)

+-----------------+-----+----------+
|         features|label|prediction|
+-----------------+-----+----------+
|[4.4,3.0,1.3,0.2]|  0.0|       0.0|
|[4.6,3.2,1.4,0.2]|  0.0|       0.0|
|[4.8,3.0,1.4,0.3]|  0.0|       0.0|
|[4.8,3.1,1.6,0.2]|  0.0|       0.0|
|[4.9,3.0,1.4,0.2]|  0.0|       0.0|
|[5.0,2.3,3.3,1.0]|  1.0|       1.0|
|[5.0,3.5,1.3,0.3]|  0.0|       0.0|
|[5.0,3.5,1.6,0.6]|  0.0|       0.0|
|[5.1,3.3,1.7,0.5]|  0.0|       0.0|
|[5.1,3.4,1.5,0.2]|  0.0|       0.0|
+-----------------+-----+----------+
only showing top 10 rows



#### 8.3 Random forest prediction

In [33]:
forest_predictions.select("features","label","prediction").show(10)

+-----------------+-----+----------+
|         features|label|prediction|
+-----------------+-----+----------+
|[4.4,3.0,1.3,0.2]|  0.0|       0.0|
|[4.6,3.2,1.4,0.2]|  0.0|       0.0|
|[4.8,3.0,1.4,0.3]|  0.0|       0.0|
|[4.8,3.1,1.6,0.2]|  0.0|       0.0|
|[4.9,3.0,1.4,0.2]|  0.0|       0.0|
|[5.0,2.3,3.3,1.0]|  1.0|       1.0|
|[5.0,3.5,1.3,0.3]|  0.0|       0.0|
|[5.0,3.5,1.6,0.6]|  0.0|       0.0|
|[5.1,3.3,1.7,0.5]|  0.0|       0.0|
|[5.1,3.4,1.5,0.2]|  0.0|       0.0|
+-----------------+-----+----------+
only showing top 10 rows



## 9.0 Comparative analysis

#### 9.1 Performance comparison using evaluation metrics
+ The results show that Logistic Regression and Random Forest achieved identical and the highest performance, while Decision Tree performed lower across all metrics

In [35]:
import pandas as pd

data = {
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Accuracy": [0.9655, 0.9310, 0.9655],
    "Precision": [0.9690, 0.9310, 0.9690],
    "Recall": [0.9655, 0.9310, 0.9655],
    "F1-score": [0.9651, 0.9310, 0.9651]
}

df = pd.DataFrame(data)
df

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.9655,0.969,0.9655,0.9651
1,Decision Tree,0.9310,0.931,0.9310,0.9310
2,Random Forest,0.9655,0.969,0.9655,0.9651


#### 9.2 Strengths and limitations of each model
+ Logistic Regression
  + Strengths:
    + High accuracy and consistent performance across all metrics
    + Simple and computationally efficient
    + Works well for linearly separable data such as Iris
  + Limitations:
    + Assumes linear relationship between features and classes
    + Less flexible for complex nonlinear patterns
+ Decision Tree
  + Strengths:
    + Easy to interpret and visualize
    + Handles nonlinear relationships
    + No need for feature scaling
  + Limitations:
    + Lower performance compared to other models
    + Prone to overfitting if not properly tuned
    + Less stable (small data changes can affect structure)
+ Random Forest
  + Strengths:
    + High accuract similar to Logistic Regression
    + More robust due to ensemble learning
    + Reduces overfitting compared to a single Decision Tree
  + Limitations:
    + More computationally expensive
    + Less interpretable compared to a single Decision Tree

#### 9.3 Justification of the best-performing model
+ Based on the evaluation results, Logistic Regression and Random Forest demonstrate the best performance as both models achieved the highest and identical values across all evaluation metrics. This indicates that both models are highly effective in correctly classifying the Iris species.
+ However, Random Forest can be considered slightly more robust in practice due to its ensemble learning approach, which combines multiple decision trees to improve generalization and reduce the risk of overfitting. In contrast, Logistic Regression performs well because the Iris dataset is relatively simple and linearly separable, making it suitable for linear classification methods.
+ Although Decision Tree is interpretable and easy to implement, its lower performance suggests that it is more sensitive to data variation and less stable compared to the other two models.
+ Overall, Random Forest is justified as the most reliable model due to its balance of high accuracy and strong generalization ability.